# V4-4-0 Portfolio

อัปเดตจาก `V3-4_portfolio` ให้ตรงกับ pipeline ใหม่ (`v4-3-0_inferencing`):
- **Data source**: รับ arrays จาก inferencing loop โดยตรง (`all_prices_gen`, `all_prices_real`, `all_log_ret_gen`, `all_log_ret_real`, `all_dates`) แทน `SimulationDataset` / `.npz`
- **Tensor dims**: shape ใหม่ `(N, W, A, C)` แทน `(N, A, W, D, C)` — A และ W สลับกัน, ไม่มี D
- **No GBM**: ไม่มี `simulate_mc_gbm` เป็น benchmark อีกแล้ว — มีแค่ `"model"` strategy เดียว
- **Imports**: ใช้ `load_data` / `build_datasets` (utils/io.py) แทน `load_variant`

# 1. Import & Config

In [20]:
import sys, os, json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import quantstats as qs
from pathlib import Path

from config import InferenceConfig, DDPMTransformerConfig, FMConfig
from utils import setup_logging, setup_random_seed, load_data, build_datasets
from utils.paths import PROCESSED_DIR, EXPERIMENTS_DIR
from entities import Portfolio

logger = setup_logging()
logger.info("✓ Imports OK")

2026-07-25 19:39:27,190 - utils.setup - INFO - Logger is set up.
2026-07-25 19:39:27,191 - utils.setup - INFO - ✓ Imports OK


## 1.1 Config

In [21]:
# 1. Import & Config
import sys, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from config import InferenceConfig
from utils import setup_logging, setup_random_seed

logger = setup_logging()
logger.info("✓ Imports OK")

# ── 1.1 Config ────────────────────────────────────────────────────────
cfg = InferenceConfig(
    name             = "portfolio_config",
    # group            = "set_index",
    # subgroup         = "set_idx50",
    # experiment_name  = "v1_trial01_w20a33_ddpm_warmup-cosine_optuna",
    # num_simulations  = 1000,

    group            = "sp_index",
    subgroup         = "sp_idx100",
    experiment_name  = "v1_trial01_w20a88_ddpm_warmup-cosine_optuna",
    num_simulations  = 100,

    checkpoint_file  = "best_model.pt",
    random_seed      = 72,
)
cfg = cfg.apply_experiment_config()

SPLIT = "test" # เปลี่ยนเป็น 'test' ได้ตามที่เซฟมา
sim_dir = cfg.experiment_dir / "simulations" / f"{SPLIT}_all_n{cfg.num_simulations}"

setup_random_seed(cfg.random_seed)

print("=" * 60)
print("  V4-4-0 PORTFOLIO")
print("=" * 60)
print(f"  Experiment      : {cfg.experiment_name}")
print(f"  Simulation dir  : {sim_dir}")
print("=" * 60)

2026-07-25 19:39:27,615 - utils.setup - INFO - Logger is set up.
2026-07-25 19:39:27,616 - utils.setup - INFO - ✓ Imports OK
[InferenceConfig] model_params loaded from best_params.json
  V4-4-0 PORTFOLIO
  Experiment      : v1_trial01_w20a88_ddpm_warmup-cosine_optuna
  Simulation dir  : /home/narodom.y@FUSION.LAB/research/02_experiments/sp_index/sp_idx100/v1_trial01_w20a88_ddpm_warmup-cosine_optuna/simulations/test_all_n100


# 3. Portfolio Optimization Loop

In [22]:
# 2. Load Simulation Data (Direct Array Load)
npz_path = sim_dir / "result_all_windows.npz"

if not npz_path.exists():
    raise FileNotFoundError(f"Not found: {npz_path} (รัน inferencing หรือยัง?)")

data = np.load(npz_path, allow_pickle=True)

# โหลด arrays (T = จำนวน window, N = sims, W = window_size, A = assets, C = channels)
all_prices_gen   = data["prices_gen"]       # (T, N, W, A)
all_prices_real  = data["prices_real"]      # (T, W, A)
all_log_ret_gen  = data["log_ret_gen"]      # (T, N, W, A, C)
all_log_ret_real = data["log_ret_real"]     # (T, W, A, C)
all_dates        = data["dates"]            # (T, W)
tickers          = data["tickers"]          # (A,)

T = all_prices_gen.shape[0]
N = all_prices_gen.shape[1]
W = all_prices_gen.shape[2]
A = all_prices_gen.shape[3]

print(f"  ✓ Loaded 1 file from {SPLIT}_all_n{cfg.num_simulations}")
print(f"  all_prices_gen   : {all_prices_gen.shape}")
print(f"  all_prices_real  : {all_prices_real.shape}")
print(f"  all_log_ret_gen  : {all_log_ret_gen.shape}")
print(f"  all_log_ret_real : {all_log_ret_real.shape}")
print(f"  all_dates        : {all_dates.shape}")
print(f"  tickers          : {len(tickers)} assets")

  ✓ Loaded 1 file from test_all_n100
  all_prices_gen   : (50, 100, 20, 88)
  all_prices_real  : (50, 20, 88)
  all_log_ret_gen  : (50, 100, 20, 88, 4)
  all_log_ret_real : (50, 20, 88, 4)
  all_dates        : (50,)
  tickers          : 88 assets


In [23]:
# 2.2 Prepare GT & MC Returns/Prices
# แปลง (T, N, W, A) -> (N, T*W, A) สำหรับ MC
# แปลง (T, W, A) -> (T*W, A) สำหรับ GT

CLOSE_IDX = 0  # เปลี่ยนให้ตรงกับ features_lr ของคุณ (0 = Close ปกติใน v4)

# ── 1. Ground Truth (GT) ──────────────────────────────────────────────
# log_ret_real : (T, W, A, C) -> เอาแค่ channel Close
gt_returns = all_log_ret_real[..., CLOSE_IDX] # (T, W, A)
gt_returns = gt_returns.reshape(-1, A)        # (T*W, A)

# prices_real : (T, W, A) ไม่มี C แล้ว
gt_prices = all_prices_real.reshape(-1, A)    # (T*W, A)

# dates : (T, W) -> flatten เป็น 1D (T*W)
if all_dates.ndim == 2:
    gt_dates = all_dates.flatten()
else:
    gt_dates = all_dates

# ── 2. Monte Carlo (MC) ───────────────────────────────────────────────
# log_ret_gen : (T, N, W, A, C) -> เอาแค่ channel Close
mc_returns = all_log_ret_gen[..., CLOSE_IDX]  # (T, N, W, A)
mc_returns = mc_returns.transpose(1, 0, 2, 3) # ดึง N มาข้างหน้าสุด (N, T, W, A)
mc_returns = mc_returns.reshape(N, -1, A)     # ยุบรวมแกนเวลา -> (N, T*W, A)

# prices_gen : (T, N, W, A)
mc_prices = all_prices_gen.transpose(1, 0, 2, 3) # (N, T, W, A)
mc_prices = mc_prices.reshape(N, -1, A)          # (N, T*W, A)

print("── Shapes after preparation ───────────────────────")
print(f"  gt_returns  : {gt_returns.shape}   (n_periods, n_assets)")
print(f"  gt_prices   : {gt_prices.shape}   (n_periods, n_assets)")
print(f"  gt_dates    : {gt_dates.shape}")
print(f"  mc_returns  : {mc_returns.shape}  (n_sims, n_periods, n_assets)")
print(f"  mc_prices   : {mc_prices.shape}   (n_sims, n_periods, n_assets)")
print("───────────────────────────────────────────────────")

── Shapes after preparation ───────────────────────
  gt_returns  : (1000, 88)   (n_periods, n_assets)
  gt_prices   : (1000, 88)   (n_periods, n_assets)
  gt_dates    : (50,)
  mc_returns  : (100, 1000, 88)  (n_sims, n_periods, n_assets)
  mc_prices   : (100, 1000, 88)   (n_sims, n_periods, n_assets)
───────────────────────────────────────────────────


In [24]:
CLOSE_IDX = 0  # channel สำหรับราคา Close

# 1. Ground Truth (GT): จับ T windows มาเรียงต่อกันให้เป็น n_periods ยาวๆ (T*W)
gt_returns = all_log_ret_real[..., CLOSE_IDX] # (T, W, A)
gt_returns = gt_returns.reshape(-1, A)        # กลายเป็น (T*W, A) แบบเส้นเดียว

gt_prices = all_prices_real.reshape(-1, A)    # กลายเป็น (T*W, A) แบบเส้นเดียว

# จัดการ Date Index: เนื่องจากไม่มีการ Overlap เราสามารถสร้างตารางเวลาเรียงต่อกันได้เลย
all_dates_flat = []
for i in range(T):
    # รองรับเผื่อเป็น scalar 1D หรือ 2D
    date_val = all_dates[i] if np.isscalar(all_dates[i]) else all_dates[i, -1]
    dates_w = pd.bdate_range(end=pd.Timestamp(date_val), periods=W)
    all_dates_flat.append(dates_w)

datetime_index = pd.DatetimeIndex(np.concatenate(all_dates_flat))

# 2. Monte Carlo (MC): ยุบรวมแกน T กับ W เข้าด้วยกัน
mc_returns = all_log_ret_gen[..., CLOSE_IDX]  # (T, N, W, A)
mc_returns = mc_returns.transpose(1, 0, 2, 3) # ดึง N ออกมาหน้าสุด (N, T, W, A)
mc_returns = mc_returns.reshape(N, -1, A)     # กลายเป็น (N, T*W, A) แบบเส้นเดียว

print("── Target Shapes for Portfolio ────────────────────")
print(f"  gt_returns : {gt_returns.shape}   (n_periods = {T*W}, n_assets = {A})")
print(f"  gt_prices  : {gt_prices.shape}   (n_periods = {T*W}, n_assets = {A})")
print(f"  datetime   : {len(datetime_index)} days")
print(f"  mc_returns : {mc_returns.shape}  (n_paths = {N}, n_periods = {T*W}, n_assets = {A})")
print("───────────────────────────────────────────────────")

── Target Shapes for Portfolio ────────────────────
  gt_returns : (1000, 88)   (n_periods = 1000, n_assets = 88)
  gt_prices  : (1000, 88)   (n_periods = 1000, n_assets = 88)
  datetime   : 1000 days
  mc_returns : (100, 1000, 88)  (n_paths = 100, n_periods = 1000, n_assets = 88)
───────────────────────────────────────────────────


In [25]:
CLOSE_FEATURE    = "Close"      # ชื่อ channel ที่ใช้ reconstruct ราคา

loaded   = load_data(str(cfg.processed_dir))
datasets = build_datasets(loaded)

tickers       = loaded["tickers"]
features_lr   = loaded["features_lr"]
features_cond = loaded["features_cond"]
scalers_x     = loaded["scalers_x"]
scalers_cond  = loaded["scalers_cond"]

CLOSE_IDX = features_lr.index(CLOSE_FEATURE)

print(f"tickers      : {tickers}")
print(f"features_lr  : {features_lr}")
print(f"close idx    : {CLOSE_IDX}")
for s in ("train", "val", "test"):
    print(f"  {s:<6} n_windows={len(datasets[s])}")

tickers      : ['AAPL', 'ABT', 'ACN', 'ADBE', 'AIG', 'ALL', 'AMGN', 'AMZN', 'AXP', 'BA', 'BAC', 'BIIB', 'BK', 'BKNG', 'BLK', 'BMY', 'C', 'CAT', 'CHTR', 'CL', 'CMCSA', 'COF', 'COP', 'COST', 'CSCO', 'CVS', 'CVX', 'DHR', 'DIS', 'DUK', 'EMR', 'EXC', 'F', 'FDX', 'GD', 'GE', 'GILD', 'GOOG', 'GOOGL', 'GS', 'HAL', 'HD', 'HON', 'IBM', 'INTC', 'JNJ', 'JPM', 'KMB', 'KO', 'LLY', 'LMT', 'LOW', 'MA', 'MCD', 'MDT', 'MET', 'MMM', 'MO', 'MRK', 'MS', 'MSFT', 'NEE', 'NFLX', 'NKE', 'NVDA', 'ORCL', 'OXY', 'PEP', 'PFE', 'PG', 'PM', 'QCOM', 'SBUX', 'SLB', 'SO', 'SPG', 'T', 'TGT', 'TXN', 'UNH', 'UNP', 'UPS', 'USB', 'V', 'VZ', 'WFC', 'WMT', 'XOM']
features_lr  : ['Close', 'High', 'Low', 'Open']
close idx    : 0
  train  n_windows=2004
  val    n_windows=246
  test   n_windows=996


In [26]:
import random
SIM_MODE = "strided"    # ← เปลี่ยนตรงนี้: "single" | "full" | "strided"

# ── config เฉพาะของแต่ละ mode ─────────────────────────────────
SINGLE_WINDOW_IDX  = None   # None → สุ่ม, หรือใส่ int ตรง ๆ เช่น 42
STRIDED_WINDOW_W   = W      # stride = W (non-overlapping), หรือตัวเลขอื่น

# ── validate ──────────────────────────────────────────────────
assert SIM_MODE in ("single", "full", "strided"), \
    f"SIM_MODE ต้องเป็น 'single' | 'full' | 'strided', ได้: {SIM_MODE!r}"

ds        = datasets[SPLIT]
n_windows = len(ds)
dates     = loaded["splits"][SPLIT]["dates"]

if SIM_MODE == "single":
    if SINGLE_WINDOW_IDX is None:
        _w = random.randint(0, n_windows - 1)
    else:
        assert 0 <= SINGLE_WINDOW_IDX < n_windows, \
            f"SINGLE_WINDOW_IDX={SINGLE_WINDOW_IDX} out of range [0, {n_windows-1}]"
        _w = SINGLE_WINDOW_IDX
    window_idxs = [_w]
    print(f"  MODE : single window")
    print(f"  idx  : {_w}  (date={dates[_w]})")

elif SIM_MODE == "full":
    window_idxs = list(range(n_windows))
    print(f"  MODE : full split ({SPLIT})")
    print(f"  windows : {n_windows}")

elif SIM_MODE == "strided":
    stride      = max(1, STRIDED_WINDOW_W)
    window_idxs = list(range(0, n_windows, stride))
    print(f"  MODE : strided (stride={stride}, W={W})")
    print(f"  windows total   : {n_windows}")
    print(f"  windows to run  : {len(window_idxs)}")

print(f"  split           : {SPLIT}")
print(f"  num_simulations : {cfg.num_simulations}")

  MODE : strided (stride=20, W=20)
  windows total   : 996
  windows to run  : 50
  split           : test
  num_simulations : 100


In [27]:
import torch
from utils import simulate_mc_gbm

print("🎲 Simulating GBM Benchmark...")

# แปลง gt_returns ที่แบนแล้ว (T*W, A) เป็น Tensor
gt_returns_tensor = torch.tensor(gt_returns, dtype=torch.float32)

# จำนวน Paths (ดึงจากขนาดของ mc_returns)
n_paths = mc_returns.shape[0]
# ความยาววันทั้งหมด
horizon_len = len(gt_returns)

# สร้าง GBM โดยใช้ mu, sigma จาก gt_returns_tensor
gbm_tensor = simulate_mc_gbm(
    log_returns=gt_returns_tensor,
    n_paths=n_paths,
    horizon=horizon_len
)

# แปลงกลับเป็น NumPy (ได้รูปทรง: [n_paths, horizon_len, n_assets])
gbm_returns = gbm_tensor.cpu().numpy()
print(f"  gbm_returns: {gbm_returns.shape}")
print("──────────────────────────────────────────────────────────────────")

🎲 Simulating GBM Benchmark...
  gbm_returns: (100, 1000, 88)
──────────────────────────────────────────────────────────────────


In [28]:
# 2. Load Simulation Data (จาก Inferencing)
npz_path = sim_dir / "result_all_windows.npz"

if not npz_path.exists():
    raise FileNotFoundError(f"ไม่พบไฟล์ {npz_path} กรุณารัน V4-3-0 Inferencing ก่อน")

data = np.load(npz_path, allow_pickle=True)

all_prices_gen   = data["prices_gen"]       # (T, N, W, A)
all_prices_real  = data["prices_real"]      # (T, W, A)
all_log_ret_gen  = data["log_ret_gen"]      # (T, N, W, A, C)
all_log_ret_real = data["log_ret_real"]     # (T, W, A, C)
all_dates        = data["dates"]            # (T, W)
tickers          = data["tickers"]          # (A,)

T = all_prices_gen.shape[0]
N = all_prices_gen.shape[1]
W = all_prices_gen.shape[2]
A = all_prices_gen.shape[3]

print(f"  ✓ Loaded 1 file from {SPLIT}_all_n{cfg.num_simulations}")
print(f"  Total Windows (T) = {T}, Simulations (N) = {N}, Window Size (W) = {W}")

  ✓ Loaded 1 file from test_all_n100
  Total Windows (T) = 50, Simulations (N) = 100, Window Size (W) = 20


In [29]:
# 3. Prepare All-Window Concatenated Data
CLOSE_IDX = 0

# 3.1 Ground Truth (GT): จับ T windows มาต่อกันเป็นเส้นเดียวยาวๆ
gt_returns = all_log_ret_real[..., CLOSE_IDX]   # (T, W, A)
gt_returns = gt_returns.reshape(-1, A)          # (T*W, A)
gt_prices = all_prices_real.reshape(-1, A)      # (T*W, A)

# 3.2 จัดการ Datetime Index
all_dates_flat = []
for i in range(T):
    date_val = all_dates[i] if np.isscalar(all_dates[i]) else all_dates[i, -1]
    dates_w = pd.bdate_range(end=pd.Timestamp(date_val), periods=W)
    all_dates_flat.append(dates_w)

datetime_index = pd.DatetimeIndex(np.concatenate(all_dates_flat))

# 3.3 Monte Carlo (MC): ดึง N ออกมาหน้าสุด แล้วยุบเวลา (T*W)
mc_returns = all_log_ret_gen[..., CLOSE_IDX]     # (T, N, W, A)
mc_returns = mc_returns.transpose(1, 0, 2, 3)    # (N, T, W, A)
mc_returns = mc_returns.reshape(N, -1, A)        # (N, T*W, A)

print(f"  ✓ gt_returns shape : {gt_returns.shape}")
print(f"  ✓ mc_returns shape : {mc_returns.shape}")
print(f"  ✓ Date Range       : {datetime_index[0].date()} to {datetime_index[-1].date()}")

  ✓ gt_returns shape : (1000, 88)
  ✓ mc_returns shape : (100, 1000, 88)
  ✓ Date Range       : 2021-12-15 to 2025-12-08


In [30]:
all_dates.shape

(50,)

In [31]:
import numpy as np
import pandas as pd
from pypfopt.efficient_frontier import EfficientFrontier
import vectorbt as vbt
import quantstats as qs
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

def run_portfolio_pipeline(
    # --- 1. ข้อมูลตั้งต้นที่ต้องโยนเข้ามา ---
    all_log_ret_gen, gt_prices, datetime_index, tickers, T, N, W,

    # --- 2. Parameter ที่ตั้งค่า Default ตามที่ต้องการ ---
    OPT_WINDOW = 20,
    REBALANCE_FREQ = 10,
    RF_RATE = 0.02,
    TRADING_FEE = 0.0025,
    TRADING_DAYS = 252,
    SOLVER = "SCS",
    ASSET_NAME = "SP100"
):
    print(f"\n{'='*60}")
    print(f"🚀 เริ่มรัน Pipeline: {ASSET_NAME} | Opt: {OPT_WINDOW} | Reb: {REBALANCE_FREQ} | Solver: {SOLVER}")
    print(f"{'='*60}\n")

    # เช็ค Logic
    if OPT_WINDOW > W:
        raise ValueError(f"🚨 ข้อผิดพลาด: OPT_WINDOW ({OPT_WINDOW}) ใหญ่กว่าขนาด Window W ({W})")

    # ==========================================
    # 0. ตั้งค่าชื่อไฟล์และสร้าง Folder
    # ==========================================
    SOLVER_NAME = SOLVER
    folder_name = f"Report_{ASSET_NAME}_Opt{OPT_WINDOW}_Rebalance{REBALANCE_FREQ}_Solver{SOLVER_NAME}"
    save_dir = Path(folder_name)
    save_dir.mkdir(parents=True, exist_ok=True)
    print(f"📁 โฟลเดอร์สำหรับบันทึกข้อมูล: {save_dir}")

    weights_filename = f"Weights_{ASSET_NAME}_Opt{OPT_WINDOW}_Rebalance{REBALANCE_FREQ}_Solver{SOLVER_NAME}.png"
    equity_filename = f"Portfolio-Equity_{ASSET_NAME}_Opt{OPT_WINDOW}_Rebalance{REBALANCE_FREQ}_Solver{SOLVER_NAME}.png"

    # ==========================================
    # 1. การเตรียมข้อมูล & วันที่ Rebalance
    # ==========================================
    close_log_ret_gen = all_log_ret_gen[:, :, :, :, 0]
    df_prices_full = pd.DataFrame(gt_prices, index=datetime_index, columns=tickers)
    df_prices_full = df_prices_full[~df_prices_full.index.duplicated(keep='last')]

    # วันที่ Rebalance
    rebalance_dates = df_prices_full.index[OPT_WINDOW - 1 :: REBALANCE_FREQ]
    if len(rebalance_dates) > T:
        rebalance_dates = rebalance_dates[:T]

    # ==========================================
    # 2. Optimization: Generative AI + MPT
    # ==========================================
    print(f"⚙️ 1/3 กำลังรัน Optimization (Generative AI + MPT)...")
    count_max_sharpe, count_min_volatility = 0, 0
    weights_list = []

    for t in range(T):
        mu_paths, cov_paths = [], []
        for n in range(N):
            path_log_returns = close_log_ret_gen[t, n, -OPT_WINDOW:, :]
            path_simple_returns = np.exp(path_log_returns) - 1

            mu_m = np.mean(path_simple_returns, axis=0) * TRADING_DAYS
            cov_m = np.cov(path_simple_returns, rowvar=False) * TRADING_DAYS
            mu_paths.append(mu_m)
            cov_paths.append(cov_m)

        mu_ensemble = np.mean(mu_paths, axis=0)
        cov_ensemble = np.mean(cov_paths, axis=0)

        mu_series = pd.Series(mu_ensemble, index=tickers)
        cov_df = pd.DataFrame(cov_ensemble, index=tickers, columns=tickers)
        cov_df = cov_df + (np.eye(len(cov_df)) * 1e-4) # ป้องกัน matrix error

        ef = EfficientFrontier(mu_series, cov_df, weight_bounds=(0, 1), solver=SOLVER)
        try:
            raw_weights = ef.max_sharpe(risk_free_rate=RF_RATE)
            count_max_sharpe += 1
        except ValueError:
            ef = EfficientFrontier(mu_series, cov_df, weight_bounds=(0, 1))
            raw_weights = ef.min_volatility()
            count_min_volatility += 1

        cleaned_weights = ef.clean_weights()
        weights_list.append(np.array([cleaned_weights[t] for t in tickers]))

    df_weights = pd.DataFrame(weights_list, index=rebalance_dates, columns=tickers)
    df_weights_full = df_weights.reindex(df_prices_full.index)

    portfolio = vbt.Portfolio.from_orders(
        close=df_prices_full, size=df_weights_full, size_type='targetpercent',
        cash_sharing=True, call_seq='auto', fees=TRADING_FEE, freq='1D'
    )

    # ==========================================
    # 3. Optimization: Equal Weight (EW)
    # ==========================================
    print(f"⚙️ 2/3 กำลังรัน Baseline (Equal Weight)...")
    df_weights_ew = pd.DataFrame(1.0 / len(tickers), index=rebalance_dates, columns=tickers)
    df_weights_ew_full = df_weights_ew.reindex(df_prices_full.index)

    portfolio_ew = vbt.Portfolio.from_orders(
        close=df_prices_full, size=df_weights_ew_full, size_type='targetpercent',
        cash_sharing=True, call_seq='auto', fees=TRADING_FEE, freq='1D'
    )

    # ==========================================
    # 4. Optimization: GBM + MPT
    # ==========================================
    print("⚙️ 3/3 กำลังรัน Baseline (GBM + MPT)...")
    gbm_weights_list = []

    for idx, reb_date in enumerate(rebalance_dates):
        loc = df_prices_full.index.get_loc(reb_date)
        hist_prices = df_prices_full.iloc[loc - OPT_WINDOW + 1 : loc + 1]
        hist_log_returns = np.log(hist_prices / hist_prices.shift(1)).dropna()

        dt = 1 / TRADING_DAYS
        mu_daily = hist_log_returns.mean().values
        cov_daily = hist_log_returns.cov().values
        num_assets = len(tickers)

        try:
            L = np.linalg.cholesky(cov_daily)
        except np.linalg.LinAlgError:
            cov_daily += np.eye(num_assets) * 1e-6
            L = np.linalg.cholesky(cov_daily)

        simulated_paths_log_ret = []
        for n in range(N):
            Z = np.random.normal(size=(W, num_assets))
            simulated_paths_log_ret.append(mu_daily + Z @ L.T)
        simulated_paths_log_ret = np.array(simulated_paths_log_ret)

        mu_paths, cov_paths = [], []
        for n in range(N):
            path_simple_returns = np.exp(simulated_paths_log_ret[n, -OPT_WINDOW:, :]) - 1
            mu_paths.append(np.mean(path_simple_returns, axis=0) * TRADING_DAYS)
            cov_paths.append(np.cov(path_simple_returns, rowvar=False) * TRADING_DAYS)

        mu_series = pd.Series(np.mean(mu_paths, axis=0), index=tickers)
        cov_df = pd.DataFrame(np.mean(cov_paths, axis=0), index=tickers, columns=tickers)

        ef = EfficientFrontier(mu_series, cov_df, weight_bounds=(0, 1), solver=SOLVER)
        try:
            raw_weights = ef.max_sharpe(risk_free_rate=RF_RATE)
        except ValueError:
            raw_weights = ef.min_volatility()

        cleaned_weights = ef.clean_weights()
        gbm_weights_list.append(np.array([cleaned_weights[t] for t in tickers]))

    df_weights_gbm = pd.DataFrame(gbm_weights_list, index=rebalance_dates, columns=tickers)
    df_weights_gbm_full = df_weights_gbm.reindex(df_prices_full.index)

    portfolio_gbm = vbt.Portfolio.from_orders(
        close=df_prices_full, size=df_weights_gbm_full, size_type='targetpercent',
        cash_sharing=True, call_seq='auto', fees=TRADING_FEE, freq='1D'
    )

    # ==========================================
    # 5. สร้าง QuantStats Report
    # ==========================================
    print(f"\n📊 กำลังสร้างและบันทึก QuantStats Reports...")
    ret_genai = portfolio.returns()
    ret_ew = portfolio_ew.returns()
    ret_gbm = portfolio_gbm.returns()

    pairs = [
        (ret_genai, ret_ew, f"Report_{ASSET_NAME}_GenAI_vs_EW_Opt{OPT_WINDOW}_Rebalance{REBALANCE_FREQ}_Solver{SOLVER_NAME}.html", f"GenAI vs EW"),
        (ret_genai, ret_gbm, f"Report_{ASSET_NAME}_GenAI_vs_GBM_Opt{OPT_WINDOW}_Rebalance{REBALANCE_FREQ}_Solver{SOLVER_NAME}.html", f"GenAI vs GBM"),
        (ret_gbm, ret_ew, f"Report_{ASSET_NAME}_GBM_vs_EW_Opt{OPT_WINDOW}_Rebalance{REBALANCE_FREQ}_Solver{SOLVER_NAME}.html", f"GBM vs EW")
    ]

    for port_ret, bench_ret, fname, title in pairs:
        fpath = str(save_dir / fname)
        qs.reports.html(returns=port_ret, benchmark=bench_ret, rf=RF_RATE,
                        title=f"{title} (Opt: {OPT_WINDOW}, Reb: {REBALANCE_FREQ})", output=fpath)
        print(f" 💾 บันทึกสำเร็จ: {fpath}")

    # ==========================================
    # 6. พล็อตกราฟ (Pie Chart, Heatmap, Equity Curve)
    # ==========================================
    print("\n🎨 กำลังวาดกราฟสรุปผล...")

    # 6.1 Pie Chart
    plt.figure(figsize=(8, 8))
    total_opts = count_max_sharpe + count_min_volatility
    plt.pie([count_max_sharpe, count_min_volatility], explode=(0.05, 0), labels=['Max Sharpe', 'Minimum Variance'],
            colors=['#4CAF50', '#FFC107'], autopct='%1.1f%%', shadow=True, startangle=90, textprops={'fontsize': 12})
    plt.title(f'Portfolio Optimization Objective Breakdown\n(Total: {total_opts} Rebalances)', fontsize=14, weight='bold', pad=30)
    plt.axis('equal')
    plt.tight_layout()
    plt.savefig(save_dir / f"Objective_PieChart_{ASSET_NAME}_Opt{OPT_WINDOW}.png", bbox_inches='tight', dpi=300)
    plt.close() # วาดเสร็จแล้วเคลียร์ออก ไม่ต้องโชว์รกหน้าจอตอนรันฟังก์ชัน

    # 6.2 Heatmap
    fig_height = max(18, len(tickers) * 0.6)
    fig, axes = plt.subplots(3, 1, figsize=(16, fig_height), sharex=True)

    dfs = [df_weights.T, df_weights_gbm.T, df_weights_ew.T]
    titles = ['1) Generative AI + MPT', '2) GBM + MPT', '3) Equal Weight (EW)']
    vmaxs = [1.0, 1.0, 0.10]

    for i, (df, ax) in enumerate(zip(dfs, axes)):
        df.columns = df.columns.strftime('%Y-%m-%d')
        sns.heatmap(df, cmap='YlGnBu', linewidths=0, vmin=0, vmax=vmaxs[i],
                    cbar_kws={'label': 'Weight Allocation', 'format': '%.3f', 'shrink': 0.8}, ax=ax)
        ax.set_title(titles[i], fontsize=14, fontweight='bold')
        ax.tick_params(axis='y', labelsize=8, rotation=0)
        if i == 2: ax.set_xlabel('Rebalance Date (Year-Month)', fontsize=12)
        ax.tick_params(axis='x', rotation=90)

    plt.suptitle('Portfolio Weights Heatmap Comparison', fontsize=18, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(save_dir / weights_filename, bbox_inches='tight', dpi=300)
    plt.close()

    # 6.3 Equity Curve
    eq_model, eq_gbm, eq_ew = portfolio.value() / portfolio.value().iloc[0], portfolio_gbm.value() / portfolio_gbm.value().iloc[0], portfolio_ew.value() / portfolio_ew.value().iloc[0]
    plt.figure(figsize=(14, 7))
    plt.plot(eq_model.index, eq_model, label=f'GenAI + MPT (Total: {(eq_model.iloc[-1] - 1)*100:.2f}%)', color='#dd8452', lw=2)
    plt.plot(eq_gbm.index, eq_gbm, label=f'GBM + MPT (Total: {(eq_gbm.iloc[-1] - 1)*100:.2f}%)', color='#55a868', lw=1.8)
    plt.plot(eq_ew.index, eq_ew, label=f'Equal Weight (Total: {(eq_ew.iloc[-1] - 1)*100:.2f}%)', color='#4c72b0', lw=1.5, ls='--')
    plt.title('Overall Portfolio Equity Curve Comparison', fontweight='bold', fontsize=16)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Portfolio Value (Base 1.0)', fontsize=12)
    plt.legend(loc='upper left', fontsize=12)
    plt.grid(True, alpha=0.3, ls=':')
    plt.margins(x=0.01)
    plt.tight_layout()
    plt.savefig(save_dir / equity_filename, bbox_inches='tight', dpi=300)
    plt.close()

    print(f"\n🎉 รันเสร็จสิ้น! ข้อมูลทั้งหมดถูกบันทึกไว้ที่: {save_dir}")

    # รีเทิร์นค่ากลับไปเผื่ออยากเอาไปดึงค่าต่อข้างนอก
    return portfolio, portfolio_gbm, portfolio_ew

In [32]:
# rebalance_freqs = [1, 5, 10, 20]
rebalance_freqs = [20]
OPT_WINDOW = 20
for rebalance_freq in rebalance_freqs:
    port_genai, port_gbm, port_ew = run_portfolio_pipeline(
        all_log_ret_gen = all_log_ret_gen,
        gt_prices = gt_prices,
        datetime_index = datetime_index,
        tickers = tickers,
        T = T,
        N = N,
        W = W,
        OPT_WINDOW=OPT_WINDOW,
        REBALANCE_FREQ=rebalance_freq,
        ASSET_NAME="SP100",
        SOLVER="OSQP",
    )


🚀 เริ่มรัน Pipeline: SP100 | Opt: 20 | Reb: 20 | Solver: OSQP

📁 โฟลเดอร์สำหรับบันทึกข้อมูล: Report_SP100_Opt20_Rebalance20_SolverOSQP
⚙️ 1/3 กำลังรัน Optimization (Generative AI + MPT)...


OptimizationError: ('Please check your objectives/constraints or use a different solver.', 'Solver status: user_limit')